In [ ]:
!pip install openai langchain langchain-openai
!pip install langchain-community langchain-text-splitters
!pip install faiss-cpu pypdf koreanize-matplotlib

In [ ]:
# ============================================================
# 실습 3-1. 규정 PDF RAG 파이프라인 (단일/멀티 소스)
# 목표: PDF 로드 -> 청킹 -> 임베딩 -> 검색 -> 생성 전 과정을 체험한다
# 실습 문서: 여비규정.pdf / 급여규정.pdf / 창업지원규정.pdf
# 사전 설치: pip install openai langchain langchain-openai
#            langchain-community langchain-text-splitters
#            faiss-cpu pypdf koreanize-matplotlib
# ============================================================

import os
import matplotlib.pyplot as plt
import koreanize_matplotlib
plt.rcParams['axes.unicode_minus'] = False
from collections import Counter

# ── API 키 설정 ──────────────────────────────────────────────
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("Colab Secrets에서 API 키 로드 완료")
except Exception:
    pass
# os.environ["OPENAI_API_KEY"] = "sk-..."

if not os.environ.get("OPENAI_API_KEY"):
    raise ValueError(
        "API 키가 설정되지 않았습니다.\n"
        "방법 A: 왼쪽 사이드바 열쇠 아이콘 -> 'OPENAI_API_KEY' 등록\n"
        "방법 B: os.environ['OPENAI_API_KEY'] = 'sk-...' 주석 해제"
    )

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
print(f"임베딩 모델 준비 완료")


# ============================================================
# [실습 변수] 여기만 수정하여 다양한 실험 가능
# ============================================================

# 기본 청크 크기: 규정집은 조항 단위가 의미의 최소 단위
# 너무 작으면 조항이 잘려 검색 품질 저하 (실습 3-1에서 직접 확인)
CHUNK_SIZE    = 400
CHUNK_OVERLAP = 50   # 청크 경계에서 정보 손실 방지용 중복

# 단일 소스 검색 청크 수
K_SINGLE = 3

# 멀티 소스 검색 청크 수 (더 많은 소스를 참조하므로 k를 높게 설정)
K_MULTI  = 5

# 청크 크기 비교 실험 값 (단계 3 시연)
# 두 값을 비교해 규정집에서 적절한 크기를 직접 확인
CHUNK_COMPARE = [100, 400]

# 단일 소스 질의 목록 {규정명: 질문}
SINGLE_QUERIES = {
    "여비규정":    "국내 출장 일비는 얼마인가요?",
    "급여규정":    "연봉 협상 주기는 어떻게 되나요?",
    "창업지원규정": "창업 지원금 신청 요건은 무엇인가요?",
}

# 멀티 소스 교차 질의 목록 (3개 규정을 동시에 참조해야 하는 질문)
MULTI_QUERIES = [
    "출장 중 창업 관련 활동을 할 경우 여비와 지원금을 동시에 받을 수 있나요?",
    "급여와 관련된 출장 수당 규정이 있나요?",
    "창업 지원 대상자의 급여 처리는 어떻게 되나요?",
]

# 청크 크기 비교 시연용 질문
CHUNK_TEST_QUESTION = "국내 출장 식비 기준을 알려주세요."

# ============================================================
# 이하 코드는 수정 불필요
# ============================================================

# ── 공통 함수 ─────────────────────────────────────────────────
def section(title):
    print(f"\n{'='*60}")
    print(f"  {title}")
    print(f"{'='*60}")

def load_pdf(filepath):
    """PDF 로드 후 페이지 수 출력"""
    loader = PyPDFLoader(filepath)
    pages  = loader.load()
    print(f"  [{filepath}] {len(pages)} 페이지 로드")
    return pages

def build_vectorstore(docs, chunk_size=CHUNK_SIZE,
                      chunk_overlap=CHUNK_OVERLAP, label=""):
    """
    청킹 + 임베딩 + FAISS 벡터 저장소 구성
    separators: 조항 번호 패턴을 포함해 규정집 구조 보존 시도
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", "제", "조", ". ", " "],
        # 규정집 특화: 조항 단위로 분할 우선 시도
    )
    chunks = splitter.split_documents(docs)
    print(f"  [{label}] 청크 수: {len(chunks)}  (chunk_size={chunk_size})")
    vs = FAISS.from_documents(chunks, embeddings)
    return vs, chunks

def make_chain(vectorstore, k=K_SINGLE):
    """
    LCEL 기반 RAG 체인 생성
    엄격 프롬프트: 문서 근거만 사용, 없으면 '확인 불가' 응답
    -> 실습 2-5에서 배운 문서 주입 원리의 자동화 버전
    """
    retriever = vectorstore.as_retriever(search_kwargs={"k": k})
    prompt    = PromptTemplate.from_template(
        "반드시 아래 제공된 문서 내용만을 근거로 답하세요.\n"
        "문서에 없는 내용은 '제공된 문서에서 확인할 수 없습니다'라고 답하세요.\n"
        "답변 끝에 참고한 조항을 명시하세요.\n\n"
        "문서 내용:\n{context}\n\n"
        "질문: {question}\n답변:"
    )
    llm   = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    fmt   = lambda docs: "\n\n".join(d.page_content for d in docs)
    chain = (
        {"context": retriever | fmt, "question": RunnablePassthrough()}
        | prompt | llm | StrOutputParser()
    )
    return chain, retriever

def run_query(chain, retriever, question, label=""):
    """질문 실행 + 결과 출력 (답변 + 참조 청크)"""
    answer  = chain.invoke(question)
    sources = retriever.invoke(question)
    print(f"\n{'='*58}")
    if label:
        print(f"[{label}]  질문: {question}")
    print(f"{'─'*58}")
    print(f"답변:\n{answer}")
    print(f"{'─'*58}")
    print(f"참조 청크 ({len(sources)}개):")
    for i, d in enumerate(sources, 1):
        src = d.metadata.get("source", "")
        pg  = d.metadata.get("page", "?")
        print(f"  [{i}] {src} p.{pg}  |  {d.page_content[:70]}...")
    print(f"{'='*58}")
    return answer, sources


# ── 단계 1: PDF 업로드 ───────────────────────────────────────
section("단계 1. PDF 파일 업로드")
print("PDF 파일을 업로드하세요.")
print("대상: 여비규정.pdf  /  급여규정.pdf  /  창업지원규정.pdf")

try:
    from google.colab import files
    uploaded = files.upload()
    pdf_files = [f for f in uploaded.keys() if f.endswith(".pdf")]
    print(f"\n업로드 완료: {pdf_files}")
except Exception:
    pdf_files = []
    print("Colab 외 환경 - PDF 파일을 현재 디렉토리에 직접 넣어주세요.")

# 파일명 자동 탐색: 키워드 포함 여부로 매핑
def find_pdf(keyword):
    for f in pdf_files:
        if keyword in f:
            return f
    return None

pdf_map = {
    "여비규정":    find_pdf("여비"),
    "급여규정":    find_pdf("급여"),
    "창업지원규정": find_pdf("창업"),
}
print(f"\n파일 매핑 결과: {pdf_map}")


# ── 단계 2: 단일 PDF 개별 호출 ──────────────────────────────
section("단계 2. 단일 PDF 개별 호출")

vs_single = {}
for name, path in pdf_map.items():
    if path is None:
        print(f"\n  [{name}] 파일 없음 - 건너뜀")
        continue
    print(f"\n-- {name} --")
    docs = load_pdf(path)
    vs, _ = build_vectorstore(docs, chunk_size=CHUNK_SIZE, label=name)
    vs_single[name] = vs

# 단일 소스 질의 실행
for name, question in SINGLE_QUERIES.items():
    if name not in vs_single:
        continue
    chain, ret = make_chain(vs_single[name], k=K_SINGLE)
    run_query(chain, ret, question, label=f"{name} 단독")

# 단일 소스 한계 설명
print("\n[단일 소스 분석]")
print("-> 각 규정 파일을 독립 벡터 저장소로 구성")
print("-> 해당 규정 내에서만 검색 가능 - 규정 간 교차 질의 불가")
print("-> '확인할 수 없습니다' 원인: PDF 표 구조 텍스트 추출 한계")
print("   (금액 정보가 표에 있어 텍스트 추출 시 구조 손실)")


# ── 단계 3: 멀티 PDF 동시 호출 ──────────────────────────────
section("단계 3. 멀티 PDF 동시 호출 (통합 벡터 저장소)")

# 3개 PDF를 하나의 벡터 저장소로 통합
# 핵심: 각 청크에 출처 메타데이터(source) 추가 -> 답변 후 출처 추적 가능
all_docs = []
for name, path in pdf_map.items():
    if path is None:
        continue
    docs = load_pdf(path)
    for d in docs:
        # 메타데이터에 출처 이름 추가: 검색 후 "여비규정 p.3" 형태로 표시
        d.metadata["source"] = name
    all_docs.extend(docs)

print(f"\n전체 로드: {len(all_docs)} 페이지")
vs_multi, all_chunks = build_vectorstore(
    all_docs, chunk_size=CHUNK_SIZE, label="통합(3개 규정)"
)

# 멀티 소스 교차 질의 실행
chain_m, ret_m = make_chain(vs_multi, k=K_MULTI)
for q in MULTI_QUERIES:
    run_query(chain_m, ret_m, q, label="멀티 소스")

print("\n[멀티 소스 핵심 관찰]")
print("-> 3개 규정 청크가 동시에 참조됨 (단일 소스에서는 불가)")
print("-> 참조 청크의 source 메타데이터로 어느 규정에서 왔는지 확인 가능")
print("-> 교차 질의(여비+창업, 급여+여비)도 통합 저장소에서 자동 수행")


# ── 단계 4: 청크 크기 비교 시연 ─────────────────────────────
section("단계 4. 청크 크기 변경 시연 (여비규정 기준)")

if pdf_map.get("여비규정"):
    docs_y = load_pdf(pdf_map["여비규정"])

    for cs in CHUNK_COMPARE:
        print(f"\n--- chunk_size = {cs} ---")
        vs_test, chunks_test = build_vectorstore(
            docs_y, chunk_size=cs, label=f"chunk_size={cs}"
        )
        chain_t, ret_t = make_chain(vs_test, k=3)
        run_query(chain_t, ret_t, CHUNK_TEST_QUESTION,
                  label=f"chunk_size={cs}")

    print("\n[청크 크기 비교 분석]")
    print(f"  chunk_size={CHUNK_COMPARE[0]}: 청크 수 많음, 조항이 잘려 맥락 손실")
    print(f"  chunk_size={CHUNK_COMPARE[1]}: 청크 수 적음, 조항 전체 포함 가능")
    print("-> 규정집은 조항 단위가 의미 최소 단위 -> 300~500 권장")
    print("   너무 작으면 '제5조 (출장 일비)' 앞뒤가 다른 청크로 분리됨")


# ── 단계 5: 통합 벡터 저장소 청크 분포 시각화 ──────────────
section("단계 5. 통합 벡터 저장소 규정별 청크 분포")

source_counts = Counter(
    d.metadata.get("source", "unknown") for d in all_chunks
)
labels  = list(source_counts.keys())
counts  = list(source_counts.values())
colors  = ["#B5D4F4", "#9FE1CB", "#FAC775"][:len(labels)]
total   = sum(counts)

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(labels, counts, color=colors, edgecolor="none", width=0.5)
for bar, cnt in zip(bars, counts):
    pct = cnt / total * 100
    ax.text(bar.get_x() + bar.get_width()/2, cnt + 1,
            f"{cnt}개\n({pct:.1f}%)",
            ha='center', va='bottom', fontsize=10)
ax.set_ylabel("청크 수")
ax.set_title(
    f"통합 벡터 저장소 - 규정별 청크 분포\n"
    f"(chunk_size={CHUNK_SIZE}, 전체 {total}개)"
)
ax.set_ylim(0, max(counts) * 1.3)
plt.tight_layout()
plt.savefig("chunk_distribution.png", dpi=150, bbox_inches='tight')
plt.show()

print("\n[청크 분포 해석]")
for name, cnt in source_counts.items():
    pct = cnt / total * 100
    print(f"  {name:12s}: {cnt:3d}개 ({pct:.1f}%)")
print("\n-> 분량이 많은 규정일수록 청크 수가 많아 검색 시 더 많이 참조됨")
print("-> 여비규정이 58%로 가장 많음 - 복잡한 출장 규정 반영")
print("\n강사 데모 완료.")
print("다음 단계: NotebookLM 실습 (본인 문서로 동일 흐름 체험)")
print("종합 실습: RFP PDF로 제안서 자동 작성 파이프라인 구현")